## 영어 -> 한국어 번역 모델

#### 데이터셋 준비

- 데이터셋: opus-100
- 데이터셋 크기: 1M 문정 썽 중에서 10만 쌍만 이용

In [3]:
# 파이썬 기본 import
import os
import json
import random
from pathlib import Path

In [4]:
# 데이터셋 마련을 위해 필요한 import
import torch
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.processors import TemplateProcessing

/opt/anaconda3/envs/playground/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
print("Pytorch: {}".format(torch.__version__))
print("MPS 사용 가능 여부: {}".format(torch.backends.mps.is_available()))

Pytorch: 2.5.1
MPS 사용 가능 여부: True


In [6]:
# 램 32GB 기준 권장 파라미터
NUM_SAMPLES = 100_000 # 전체 데이터셋은 10만개 쌍만 이용
MAX_LEN = 128 # 최대 토큰의 길이
VOCAB_SIZE = 16_000 # BPE Vocab 의 사이즈
MIN_FREQ = 2 # vocab 최소 빈도
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.8, 0.1, 0.1 # train : validation : test = 8 : 1 : 1
BATCH_SIZE = 32
SEED = 42
DATA_DIR = Path("data")

In [7]:
# 특수 토큰 정의
PAD_TOKEN = "[PAD]";  PAD_ID = 0
UNK_TOKEN = "[UNK]";  UNK_ID = 1
BOS_TOKEN = "[BOS]";  BOS_ID = 2
EOS_TOKEN = "[EOS]";  EOS_ID = 3
SPECIAL_TOKENS = [PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOS_TOKEN]

DATA_DIR.mkdir(exist_ok=True)
print("설정 완료")

설정 완료


In [8]:
# 데이터셋 로드
raw_dataset = load_dataset("Helsinki-NLP/opus-100", "en-ko", trust_remote_code=True)
print(raw_dataset)

DatasetDict({
    test: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
    train: Dataset({
        features: ['translation'],
        num_rows: 1000000
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
})


In [9]:
# 샘플링
pairs = [
    {
        "en" : item["translation"]["en"],
        "ko" : item["translation"]["ko"]
    } for item in raw_dataset["train"]
]

print("전체 문장 쌍 개수 : {}".format(len(pairs)))

전체 문장 쌍 개수 : 1000000


In [10]:
# 샘플 확인
for p in pairs[:3]:
    print("EN: {}".format(p["en"]))
    print("KO: {}".format(p["ko"]))
    print()

EN: They're shaped like a bus.
KO: 할머니처럼 만들었지만.. ? 엉망이지만..

EN: I ain't fishing' 'em out.
KO: 그거 꺼내려다가는

EN: You are torturing god's creatures in an age where we have the technology that no longer requires us to.
KO: 선생님은 이 기술력이 있는 시대에 그러지 않아도 되는데도 신의 피조물을 괴롭히고 있다고요



In [11]:
# 서브셋 샘플링
if NUM_SAMPLES and NUM_SAMPLES < len(pairs):
    random.seed(SEED)
    pairs = random.sample(pairs, NUM_SAMPLES)
    print("서브셋 샘플링 완료 : {} 문장쌍".format(len(pairs)))

서브셋 샘플링 완료 : 100000 문장쌍


In [12]:
def is_valid(en: str, ko: str, max_words: int = 80) -> bool:
    en, ko = en.strip(), ko.strip()

    # 영어 문장 혹은 한국어 문장이 존재하지 않은 경우
    if not en or not ko:
        return False

    # 영어 문장 혹은 한국어 문장이 한 문장으로 종결되는 경우
    if len(en.split()) < 2 or len(ko.split()) < 2:
        return False

    if len(en.split()) > max_words or len(ko.split()) > max_words:
        return False

    return True

In [13]:
before = len(pairs)
pairs = [p for p in pairs if is_valid(p["en"], p["ko"])]
print(f"필터 전 : {before:,}  →  필터 후 : {len(pairs):,}  (제거 : {before - len(pairs):,})")

필터 전 : 100,000  →  필터 후 : 86,625  (제거 : 13,375)


In [14]:
# 기본 토크나이저 설정 시작
TOK_DIR = DATA_DIR / "tokenizer"
TOK_PATH = TOK_DIR / "tokenizer.json"
TOK_DIR.mkdir(exist_ok=True)

tokenizer = Tokenizer(BPE(unk_token=UNK_TOKEN))
tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=MIN_FREQ,
    special_tokens=SPECIAL_TOKENS,
    show_progress=True
)

In [15]:
def sentence_iter():
    for p in pairs:
        yield p["en"]
        yield p["ko"]

In [16]:
tokenizer.train_from_iterator(sentence_iter(), trainer=trainer)

In [17]:
# 인코딩 시 BOS/EOS 자동 추가
tokenizer.post_processor = TemplateProcessing(
    single=f"{BOS_TOKEN} $A {EOS_TOKEN}",
    special_tokens=[(BOS_TOKEN, BOS_ID), (EOS_TOKEN, EOS_ID)],
)

In [18]:
tokenizer.save(str(TOK_PATH))
print(f"Tokenizer 저장 -> {TOK_PATH}")
print(f"Vocab 크기 : {tokenizer.get_vocab_size():,}")

Tokenizer 저장 -> data/tokenizer/tokenizer.json
Vocab 크기 : 16,000


In [19]:
# 토크나이저 동작 확인
sample_en = "The model learns to translate English to Korean"
sample_ko = "모델은 영어를 한국어로 번역하는 방법을 학습합니다."

enc_en = tokenizer.encode(sample_en)
enc_ko = tokenizer.encode(sample_ko)

print(f"EN  ids  : {enc_en.ids}")
print(f"EN  toks : {enc_en.tokens}")
print()
print(f"KO  ids  : {enc_ko.ids}")
print(f"KO  toks : {enc_ko.tokens}")

EN  ids  : [2, 2100, 14411, 3854, 3274, 2045, 3744, 3428, 9634, 2045, 15172, 3]
EN  toks : ['[BOS]', 'The', 'model', 'lear', 'ns', 'to', 'trans', 'late', 'English', 'to', 'Korean', '[EOS]']

KO  ids  : [2, 14695, 1461, 1387, 5868, 14241, 7308, 9516, 2139, 5851, 1930, 1249, 2436, 17, 3]
KO  toks : ['[BOS]', '모델', '은', '영', '어를', '한국', '어로', '번역', '하는', '방법을', '학', '습', '합니다', '.', '[EOS]']


In [20]:
# 커스텀 데이터 토크나이징
tokenizer.enable_truncation(max_length=MAX_LEN)

tokenized, skipped = [], 0
for p in pairs:
    src_ids = tokenizer.encode(p["en"]).ids
    tgt_ids = tokenizer.encode(p["ko"]).ids

    if len(src_ids) > MAX_LEN or len(tgt_ids) > MAX_LEN:
        skipped += 1
        continue

    tokenized.append({
        "src": src_ids,
        "tgt": tgt_ids
    })

print(f"토크나이즈 완료: {len(tokenized):,} (길이 초과 제거 : {skipped:,})")

토크나이즈 완료: 86,625 (길이 초과 제거 : 0)


In [21]:
random.seed(SEED)
random.shuffle(tokenized)

In [22]:
n = len(tokenized)
n_train, n_val = int(n * TRAIN_RATIO), int(n * VAL_RATIO)

train_data = tokenized[:n_train]
val_data = tokenized[n_train : n_train + n_val]
test_data = tokenized[n_train + n_val :]

print(f"Train : {len(train_data):,}")
print(f"Val   : {len(val_data):,}")
print(f"Test  : {len(test_data):,}")

Train : 69,300
Val   : 8,662
Test  : 8,663


In [23]:
# 가공된 데이터 저장
def save_split(data, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)

save_split(train_data, DATA_DIR / "train.json")
save_split(val_data,   DATA_DIR / "val.json")
save_split(test_data,  DATA_DIR / "test.json")

print(f"저장 완료 -> {DATA_DIR}/")

저장 완료 -> data/
